# Why our quarters differ from AMW's: the last day

AMW's quarters stop one day early. Earth Engine excludes the end date, and AMW passes the quarter's last
day as the end, so 31 Mar, 30 Jun and 30 Sep are never used. Ours end on the 1st of the next month,
so every day is in. This notebook shows it on the Pará test area, 2026Q1, by running AMW's own code with both end dates.

In [1]:
import sys
from pathlib import Path
import json
import geopandas as gpd
import pandas as pd

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "panel_pipeline").is_dir())
sys.path.insert(0, str(REPO / "core"))
from postprocess import centroid_key   # AMW's chip key: the chip's centre, rounded

RUN = REPO / "tests_output/test_region_para_2026-10-19"     # our detect.py run, subregion 4
TASK = RUN / "detect/quarterly/period=2026Q1/subregion=4"
RERUNS = REPO / "data/outputs/48px_v4.10b-18d-20g-21a-22bc-ensemble"
FILES = {
    "AMW published": REPO / "amw_comparison/data/amw/amazon_basin_Q126_t0.4.geojson",
    "AMW code, end 31 Mar": RERUNS / "test_region_pará_48px_v4.10b-18d-20g-21a-22bc-ensemble_0.40_2026-01-01_2026-03-31.geojson",
    "AMW code, end 1 Apr": RERUNS / "test_region_pará_48px_v4.10b-18d-20g-21a-22bc-ensemble_0.40_2026-01-01_2026-04-01.geojson",
}

## 1. The windows

Both end dates are *excluded* by Earth Engine: `ee.Filter.date` takes "the inclusive start date" and
"the optional exclusive end date" ([google/earthengine-api, python/ee/filter.py](https://github.com/google/earthengine-api/blob/master/python/ee/filter.py)).
AMW's file names carry the dates its code passed to Earth Engine (`core/inference_engine.py:494`).

In [2]:
status = json.loads((TASK / "_status.json").read_text())
first_line = FILES["AMW published"].open().read(300)
amw_span = first_line.split('"name": "')[1].split('"')[0].rsplit("_", 2)[-2:]
pd.DataFrame({"start": [amw_span[0], status["start_date"]],
              "end (excluded)": [amw_span[1], status["end_date_excluded"]],
              "last day used": ["30 Mar", "31 Mar"]}, index=["AMW", "ours"])

,start,end (excluded),last day used
AMW,2026-01-01,2026-03-31,30 Mar
ours,2026-01-01,2026-04-01,31 Mar


## 2. Four sets of detections, chip by chip

Detections are chips with confidence > 0.4. *identical* = same score to 1e-6 (the same images went in).

In [3]:
chips = pd.read_parquet(RUN / "plan/chips.parquet", columns=["chip_id", "amw_key_x", "amw_key_y"])
ours = pd.concat(pd.read_parquet(p, columns=["chip_id", "confidence", "n_clear"]) for p in TASK.glob("part_*.parquet"))
scores = {"ours (detect.py)": ours.set_index("chip_id")["confidence"].loc[lambda s: s > 0.4]}
for name, path in FILES.items():
    gdf = gpd.read_file(path)
    gdf["amw_key_x"], gdf["amw_key_y"] = centroid_key(gdf)
    scores[name] = chips.merge(gdf, on=["amw_key_x", "amw_key_y"]).set_index("chip_id")["confidence"]

def compare(a, b):
    sa, sb = scores[a], scores[b]
    both = sa.index.intersection(sb.index)
    gap = (sa[both] - sb[both]).abs()
    return {"A": a, "B": b, "detections A": len(sa), "detections B": len(sb), "both": len(both),
            "only A": len(sa.index.difference(sb.index)), "only B": len(sb.index.difference(sa.index)),
            "identical": f"{(gap <= 1e-6).mean():.0%}", "largest gap": round(gap.max(), 4)}

pd.DataFrame([compare("AMW published", "AMW code, end 31 Mar"),
              compare("ours (detect.py)", "AMW code, end 1 Apr"),
              compare("ours (detect.py)", "AMW published")])

,A,B,detections A,detections B,both,only A,only B,identical,largest gap
0,AMW published,"AMW code, end 31 Mar",27,27,27,0,0,100%,0.0000
1,ours (detect.py),"AMW code, end 1 Apr",56,56,56,0,0,100%,0.0000
2,ours (detect.py),AMW published,56,27,26,30,1,8%,0.2659


AMW's code with its own end date reproduces AMW's published file exactly, and with our end date it
reproduces our run exactly. So the code agrees, Earth Engine's images haven't changed since AMW ran,
and the whole gap is 31 March.

## 3. Why one day matters so much here

Q1 is the rainy season: each pixel has only a few clear looks in the whole quarter, so one clear scene
on the last day is a big share of the median image.

In [4]:
q2 = pd.concat(pd.read_parquet(p, columns=["n_clear"]) for p in (RUN / "detect/quarterly/period=2026Q2/subregion=4").glob("part_*.parquet"))
pd.Series({"2026Q1": ours["n_clear"].median(), "2026Q2": q2["n_clear"].median()}, name="median clear looks per pixel")

2026Q1     4.622830
2026Q2    16.581814
Name: median clear looks per pixel, dtype: float32

## Conclusion

We keep the last day: our quarters are whole calendar quarters, like our months. Any comparison with
AMW's published quarters differs wherever the last day had a clear scene, and those differences are
expected, not errors.